In [17]:
import os
import re
from collections import Counter

# 读取file里面的所有行, 返回列表. [第一行,第二行,第三行.]
def read_file(path):
    with open(path, 'r') as f:
        content = f.readlines()
    return content

# 2.Tokenization  利用正则表达式,把行拆成词语,返回一个列表. [词语1,词语2,词语3]
def tokenize(doc):
    rule = r'[\s\~\`\!\@\#\$\%\^\&\*\(\)\-\_\+\=\{\}\[\]\;\:\'\"\,\<\.\>\/\?\\|]+'
    re.compile(rule)  #生成一个正则表达式对象.

    terms_ = []
    for line in doc:
        terms_ = terms_ + re.split(rule, line.lower())    #按照rule把句子,分开为词语.

    terms = []
    for term in terms_:
        if term != '':
            terms.append(term)
    return terms

def processing(path):
    # read file
    doc = read_file(path)
    terms = tokenize(doc)
    term_freq = Counter(terms) #用来数词语出现频率,返回一个dict subclass
    return dict(term_freq)    #强转回,dict.


# load all files


In [18]:
root = os.path.abspath('.')

#生成term frequency. 一个单词在一个文档的,出现次数.
term_freq= [None]*10
for i in range(10):
    term_freq[i] =processing(os.path.join(root, 'documents', f'doc{i+1}.txt'))

print(term_freq)

[{'i': 4, 'love': 1, 'macau': 3, 'am': 2, 'a': 2, 'student': 1, 'studying': 1, 'in': 3, 'the': 2, 'university': 2, 'of': 2, 'um': 4, 'has': 2, 'very': 1, 'beautiful': 1, 'campus': 1, 'why': 1, 'did': 1, 'choose': 1, 'because': 1, 'great': 1, 'reputation': 1, 'education': 1, 'research': 1, 'and': 1, 'innovation': 1, 'proud': 1, 'to': 1, 'study': 1}, {'python': 2, 'is': 3, 'an': 1, 'interpreted': 1, 'high': 1, 'level': 1, 'general': 1, 'purpose': 1, 'programming': 3, 'language': 3, 'its': 5, 'design': 1, 'philosophy': 1, 'emphasizes': 1, 'code': 2, 'readability': 1, 'with': 1, 'use': 1, 'of': 1, 'significant': 1, 'indentation': 1, 'constructs': 1, 'as': 3, 'well': 1, 'object': 2, 'oriented': 2, 'approach': 1, 'aim': 1, 'to': 2, 'help': 1, 'programmers': 1, 'write': 1, 'clear': 1, 'logical': 1, 'for': 1, 'small': 1, 'and': 3, 'large': 1, 'scale': 1, 'projects': 1, 'dynamically': 1, 'typed': 1, 'garbage': 1, 'collected': 1, 'it': 2, 'supports': 1, 'multiple': 1, 'paradigms': 1, 'including'

In [19]:
# build vocabulary  extract all keys(terms) take set to remove duplicate
vocab = []
for i in range (10):
    vocab = list(set(list(term_freq[i].keys()) +vocab))
    
print(len(vocab))

687


In [20]:
#build term-doc tf-idf matrix

import numpy as np
import pandas as pd
tfidf_mat = pd.DataFrame(np.zeros((len(vocab), 10)), index = vocab)
#製造一个 dataframe  #class pandas.DataFrame(data=None, index=None, columns=None, dtype=None, copy=None)
# Return a new array of given shape and type, filled with zeros.

#把之前统计好的,term frequency,放进 dateframe
for i in range(10):
    for key,value in term_freq[i].items():
        tfidf_mat.loc[key, i] = value
        # pandas.dataframe.loc . Access a group of rows and columns by label(s) or a boolean array.
print(tfidf_mat) #弄好 term frequency了.

              0    1    2    3    4    5    6    7    8    9
changed     0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0
circuits    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0
very        1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
constructs  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
analyzing   0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0
...         ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
well        0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
inspection  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0
informed    0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0
until       0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0
usually     0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0

[687 rows x 10 columns]


In [21]:
# calculate idf
# doc_freq = df. 一个单词,在多少个文档出现过.
doc_freq = np.count_nonzero(tfidf_mat, axis=1)   # row horizon
print(doc_freq)
idf = np.log(10/(1+doc_freq))
# 水平计算每行不包括0的个数. 从而计算出,该字在多少个文档出现过. # numpy.count_nonzero(a, axis=None, *, keepdims=False)

#calculate tf-idf.
idf = np.reshape(idf, (len(idf), 1))
tfidf_mat = tfidf_mat * idf
print(tfidf_mat)
# idf是一个很长的row,转成column(一个怪格式),才能和data frame相乘.
# 元素級別乘法.

[ 1  1  1  1  1  4  1  1  1  1  9  1  1  1  1  1  1  2  1  1  1  1  2  1
  1  2  1  2  1  1  1  1  1  4  1  2  1  1  1  2  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  2  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  4  2  1  1  1  4  1  1  1  2  1  1  1  1  2  2 10  1  2  1  1  1
  1  1  1  1  1  1  1  1  1  1  4  1  1  1  1  1  1  1  1  1  1  1  5  1
  1  1  1  1  1  1  1  1  4  1  2  1  1  2  1  3  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  3  1  1  1  1  1  1  1  3  1  1
  1  1  1  1  1  1  1  2  1  1  1  1  2  1  1  2  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  2  1  1  1  2  1  1  1  1  1  1
  1  2  1  1 10  1  1  1  1  1  1  2  1  1  9  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  2  5  1  1  1  1  1  1  9  1  6  1  5  1  1  1  1
  1  1  2  4  1  1  1  1  1  1  1  1  1  1  1  1  5  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  2  1  1  1  2  1  1  1  1  1  1  1  1  1  1  1  2
  1  1  1  1  5  2  2  1  1  2  1  1  1  1  1  1  1

# query

In [22]:
query=['University of Macau','Gambling Game','American Football Game']

#把句子分开一个个单词
q_term = [None]*3
for i in range (3):
    q_term[i]= query[i].lower().split(' ')



#生成 3 个 query dataframe把 df装进去.  df只有1,而不是用counter函数,去记录.不知道为什么哈
query_tf=[None] * 3
for i in range (3):
    query_tf[i] = pd.DataFrame(np.zeros((len(vocab), 1)), index = vocab)
for i in range (3):
    for term in q_term[i]:
        query_tf[i].loc[term, 0] = 1

#计算tfidf
query_tfidf=[None]*3
for i in range (3):
    query_tfidf[i] = query_tf[i] * idf

for i in range(3):
    print(query_tfidf[i])

# cosine similarity
def cosine(a, b):
    cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    return cos_sim

              0
changed     0.0
circuits    0.0
very        0.0
constructs  0.0
analyzing   0.0
...         ...
well        0.0
inspection  0.0
informed    0.0
until       0.0
usually     0.0

[687 rows x 1 columns]
              0
changed     0.0
circuits    0.0
very        0.0
constructs  0.0
analyzing   0.0
...         ...
well        0.0
inspection  0.0
informed    0.0
until       0.0
usually     0.0

[687 rows x 1 columns]
              0
changed     0.0
circuits    0.0
very        0.0
constructs  0.0
analyzing   0.0
...         ...
well        0.0
inspection  0.0
informed    0.0
until       0.0
usually     0.0

[687 rows x 1 columns]


In [23]:
# 计算分数, 一个句子要算出10个分数,因为10个文档.
# score =[[None]*10]*3 python bug
score = [[None]*10 for _ in range(3)]

for j in range (3):
    for i in range(10):
        score[j][i] = cosine(tfidf_mat.iloc[:,i], query_tfidf[j])
documents = [f'doc{x}' for x in range (1,11)]
print(score)

[[array([0.25609208]), array([0.00047112]), array([0.15101564]), array([0.00168579]), array([0.00254185]), array([0.00167155]), array([0.00187943]), array([0.00141787]), array([0.10512846]), array([0.06723035])], [array([0.]), array([0.]), array([0.]), array([0.02329621]), array([0.]), array([0.]), array([0.]), array([0.10449991]), array([0.47951227]), array([0.])], [array([0.]), array([0.]), array([0.]), array([0.01998505]), array([0.]), array([0.03170604]), array([0.02970755]), array([0.2498428]), array([0.]), array([0.])]]


In [26]:
dictionary =[None]*3
for i in range (3):
    dictionary[i] = dict(zip(documents, score[i]))

for i in range (3):
    print(dictionary[i])
    print()

{'doc1': array([0.25609208]), 'doc2': array([0.00047112]), 'doc3': array([0.15101564]), 'doc4': array([0.00168579]), 'doc5': array([0.00254185]), 'doc6': array([0.00167155]), 'doc7': array([0.00187943]), 'doc8': array([0.00141787]), 'doc9': array([0.10512846]), 'doc10': array([0.06723035])}

{'doc1': array([0.]), 'doc2': array([0.]), 'doc3': array([0.]), 'doc4': array([0.02329621]), 'doc5': array([0.]), 'doc6': array([0.]), 'doc7': array([0.]), 'doc8': array([0.10449991]), 'doc9': array([0.47951227]), 'doc10': array([0.])}

{'doc1': array([0.]), 'doc2': array([0.]), 'doc3': array([0.]), 'doc4': array([0.01998505]), 'doc5': array([0.]), 'doc6': array([0.03170604]), 'doc7': array([0.02970755]), 'doc8': array([0.2498428]), 'doc9': array([0.]), 'doc10': array([0.])}



In [25]:
for i in range (3):
    print(query[i],"most relevant top 3 doc:")
    L = sorted(dictionary[i].items(),key=lambda item:item[1],reverse=True)
    L = L[:3]
    print(L)
    print()

University of Macau most relevant top 3 doc:
[('doc1', array([0.25609208])), ('doc3', array([0.15101564])), ('doc9', array([0.10512846]))]

Gambling Game most relevant top 3 doc:
[('doc9', array([0.47951227])), ('doc8', array([0.10449991])), ('doc4', array([0.02329621]))]

American Football Game most relevant top 3 doc:
[('doc8', array([0.2498428])), ('doc6', array([0.03170604])), ('doc7', array([0.02970755]))]



# 第二道题

(1)‘^[A-Z]\d\d\s[3-6]+’ 
[1]  ‘B11 234’     [2] ‘bB11 234’    [3] ‘bB11 23z4’ 

None of them are correct.
explain: start with A to Z,two digit in the middle , one space, at least one number in range 3-6


(2) ‘db\d{5}@um\.edu\.mo’ 
[1]  ‘db123@um.edu.mo     [2] ‘dbc123@um.edu.mo’    [3] ‘ab12345@um.edu.mo’ 
None of them are correct.
start with db, 5 character of digit, end with @um.edu.mo

(3) ‘^\-\d+\.\d{2}’ 
[1] ‘-12.1’                [2] ‘-12.10’      [3] ‘-1.210’ 

correct answer is [2] ‘-12.10’
start with - , at list one character of digit , a character of . , end with 2 digit

(4) ‘\_{2}[a-z]*\_$’  
[1] ‘_ab__’               [2] ‘__aB_’       [3] ‘__abcdfg_’

correct answer is [3] ‘__abcdfg_’ 
start with two _ ,any length of character in the range a to z , ending with one _

(5) ‘[a-z|A-Z]+12345\_’ 
[1] ‘ab12345_’        [2] ‘aB12345_’   [3] ‘1a2345_’

correct answer is [1] ‘ab12345_’        [2] ‘aB12345_’ 

start with at least one english letter, end with 12345_.